# Modern Solution API Basics

This notebook is the canonical starting point for the current `hdg_postprocess` solution API.

It shows how to:
- load a solution with `hdg_postprocess.api.load_solution`
- inspect mesh, state, and summary containers
- access assembled fields through the grouped facades
- sample profiles and pointwise diagnostics
- run one setup-heavy analysis workflow in the modern style

The older `hdg_solution_*.ipynb` notebooks are still available under `demos/compatibility/`, but new code should prefer the patterns shown here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hdg_postprocess.api import (
    configure_solution_setup,
    load_reference_element,
    load_solution,
)


## Load one demo solution

For new code, prefer `load_solution(...)` instead of the older `load_from_file.load_HDG_solution_from_file(...)` entrypoint.

In [ ]:
solution = load_solution(
    "../data/solutions/power_balance_boundary_checks/",
    "solution_west_heating_cooling_factor",
    n_partitions=1,
)

solution.mesh.metadata.reference_element = load_reference_element(
    "../data/reference_elements/reference_triangle_P8.mat"
)


## Inspect the main containers

The modern API separates stored state from behavior:
- `solution.views` stores assembled state
- `solution.summary` stores integrated outputs
- `solution.metadata` stores flags and cache status
- grouped facades such as `solution.fields`, `solution.sample`, and `solution.analysis` drive workflows

In [ ]:
solution.views
solution.summary
solution.metadata.flags


## Access mesh data explicitly

Mesh state also follows the same grouped style:
- `mesh.metadata` for descriptive information
- `mesh.global_state` for recombined arrays
- `mesh.derived_geometry` for cached geometry products
- `mesh.plot` for plotting

In [ ]:
solution.mesh.assembly.full()

vertices = solution.mesh.global_state.vertices
connectivity = solution.mesh.global_state.connectivity
connectivity_big = solution.mesh.geometry.connectivity_big

{
    "n_elements": solution.mesh.global_state.n_elements,
    "vertices_shape": vertices.shape,
    "connectivity_shape": connectivity.shape,
    "connectivity_big_shape": connectivity_big.shape,
    "extent": solution.mesh.metadata.extent,
}


## Assemble and inspect global, simple, and Gauss views

For new analysis code, prefer the grouped field facade for assembly and direct `views` access for inspection.

In [ ]:
full_cons = solution.fields.conservative(view="full")
simple_phys = solution.fields.physical(view="simple")
gauss_phys = solution.fields.physical(view="gauss")

{
    "full_cons": full_cons.shape,
    "simple_phys": simple_phys.shape,
    "gauss_phys": gauss_phys.shape,
}


In [ ]:
{
    "glob_cons_cached": solution.views.glob.solution.conservative is full_cons,
    "simple_phys_cached": solution.views.simple.solution.physical is simple_phys,
    "gauss_phys_cached": solution.views.gauss.solution.physical is gauss_phys,
}


## Equilibrium facade and cached equilibrium views

Magnetic-equilibrium workflows follow the same split:
- `solution.assembly.simple()` prepares the simple cached state
- `solution.equilibrium.*` adds derived equilibrium quantities such as the magnetic axis, minor radius, and cylindrical safety factor
- `solution.views.simple.equilibrium` exposes the containered equilibrium arrays
- `solution.summary.equilibrium.*` stores integrated equilibrium summaries


In [ ]:
solution.assembly.simple()
axis = solution.equilibrium.define_axis()
minor_radius = solution.equilibrium.define_minor_radii(view="simple")
qcyl = solution.equilibrium.define_qcyl(view="simple")

eq_magnetic_field = solution.equilibrium.magnetic_field(view="simple")
eq_poloidal_flux = solution.equilibrium.poloidal_flux(view="simple")
eq_jtor = solution.equilibrium.jtor(view="simple")

{
    "axis_r": axis.r,
    "axis_z": axis.z,
    "cached_in_summary": solution.summary.equilibrium.axis is axis,
    "magnetic_field_shape": None if eq_magnetic_field is None else eq_magnetic_field.shape,
    "poloidal_flux_shape": None if eq_poloidal_flux is None else eq_poloidal_flux.shape,
    "jtor_shape": None if eq_jtor is None else eq_jtor.shape,
    "minor_radius_shape": None if minor_radius is None else minor_radius.shape,
    "qcyl_shape": None if qcyl is None else qcyl.shape,
}


## Plot equilibrium fields from the cached simple view

Once the equilibrium state is prepared, the containered arrays in `solution.views.simple.equilibrium` can be plotted directly.


In [ ]:
eq_magnetic_field = solution.equilibrium.magnetic_field(view="simple")
eq_poloidal_flux = solution.equilibrium.poloidal_flux(view="simple")
eq_jtor = solution.equilibrium.jtor(view="simple")
btheta = None if eq_magnetic_field is None else eq_magnetic_field[:, 2]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

im0 = axes[0].tripcolor(
    solution.mesh.global_state.vertices[:, 0],
    solution.mesh.global_state.vertices[:, 1],
    solution.mesh.geometry.connectivity_big,
    eq_poloidal_flux,
    shading="flat",
)
fig.colorbar(im0, ax=axes[0], label="psi")
axes[0].set_title("Simple-view poloidal flux")
axes[0].set_aspect("equal")

if btheta is not None:
    im1 = axes[1].tripcolor(
        solution.mesh.global_state.vertices[:, 0],
        solution.mesh.global_state.vertices[:, 1],
        solution.mesh.geometry.connectivity_big,
        btheta,
        shading="flat",
    )
    fig.colorbar(im1, ax=axes[1], label="B_theta")
    axes[1].set_title("Simple-view toroidal magnetic field")
    axes[1].set_aspect("equal")
else:
    axes[1].text(0.5, 0.5, "magnetic field not available", ha="center", va="center")
    axes[1].set_title("Simple-view toroidal magnetic field")

if eq_jtor is not None:
    im2 = axes[2].tripcolor(
        solution.mesh.global_state.vertices[:, 0],
        solution.mesh.global_state.vertices[:, 1],
        solution.mesh.geometry.connectivity_big,
        eq_jtor,
        shading="flat",
    )
    fig.colorbar(im2, ax=axes[2], label="jtor")
    axes[2].set_title("Simple-view jtor")
    axes[2].set_aspect("equal")
else:
    axes[2].text(0.5, 0.5, "jtor not available in this case", ha="center", va="center")
    axes[2].set_title("Simple-view jtor")

plt.show()


## Plot one field on the recombined mesh

Plotting stays grouped behind `solution.plot` and `solution.mesh.plot`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
solution.mesh.plot.full(ax=ax, data=simple_phys[:, 0], connectivity=connectivity_big)
ax.set_title("Simple-view physical field on the refined plotting mesh")
plt.show()


## Sample a profile with the sampling facade

The line sampler is the preferred public API for profile-style interpolation.

In [ ]:
r_min, r_max = vertices[:, 0].min(), vertices[:, 0].max()
z_mid = 0.5 * (vertices[:, 1].min() + vertices[:, 1].max())

r_line = np.linspace(r_min, r_max, 200)
z_line = np.full_like(r_line, z_mid)

solution.sample.define_interpolators()
profile = solution.sample.line(r_line, z_line, ["n", "te", "ti", "M"])

list(profile.keys())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r_line, profile["n"], label="n")
ax.plot(r_line, profile["te"], label="te")
ax.plot(r_line, profile["ti"], label="ti")
ax.set_xlabel("R [m]")
ax.legend()
ax.set_title("Sampled profile along an approximate midplane line")
plt.show()


## Pointwise access is grouped semantically

Use `solution.pointwise.*` when you want one local diagnostic rather than a whole profile.

In [ ]:
finite_idx = np.where(np.isfinite(profile["n"]))[0][len(np.where(np.isfinite(profile["n"]))[0]) // 2]
r0 = float(r_line[finite_idx])
z0 = float(z_line[finite_idx])

{
    "point": (r0, z0),
    "n": solution.pointwise.plasma.n(r0, z0),
    "ti": solution.pointwise.plasma.ti(r0, z0),
    "grad_ti_x": solution.pointwise.gradients.ti(r0, z0, "x"),
    "b_theta": solution.pointwise.fields.magnetic_field(r0, z0, "theta"),
}


## One setup-heavy analysis workflow

For power balance and related workflows, use `configure_solution_setup(...)` instead of manually scattering setup through notebook cells.

In [ ]:
solution.parameters["physics"]["R_E"] = solution.parameters["physics"].get("R_E", 1.0)

configure_solution_setup(
    solution,
    reference_element="../data/reference_elements/reference_triangle_P8.mat",
    radiation_model="nitrogen_cooling",
    atomic_data_dir="../data/atomic",
    neutral_diffusion=True,
)

power = solution.analysis.power_balance()
boundary = solution.analysis.boundary_summary()

{
    "power_keys": list(power.keys())[:8],
    "boundary_keys": list(boundary.keys())[:8],
}


## Plot a divertor segment from the boundary summary

A practical boundary-summary workflow is to mark two points on the divertor-facing boundary, convert them to ordered boundary indices with `nearest_face_index(...)`, and then plot the stored boundary profile over that segment.


In [ ]:
r_low = 1.9091
z_low = -0.5798
r_high = 2.446
z_high = -0.7964

idx_low = solution.mesh.boundary.nearest_face_index(r_low, z_low)
idx_high = solution.mesh.boundary.nearest_face_index(r_high, z_high)

i0, i1 = sorted((idx_low, idx_high))
segment = slice(i0, i1)

fig, ax = plt.subplots(figsize=(8, 8))
solution.mesh.plot.full(ax=ax)
ax.scatter([r_low, r_high], [z_low, z_high], color=["red", "green"], s=80)
ax.set_title("Selected divertor segment on the full mesh")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

axes[0].plot(boundary["r"][segment].flatten(), boundary["te"][segment].flatten())
axes[0].set_title("Electron temperature")
axes[0].set_xlabel("R [m]")
axes[0].set_ylabel("Te [eV]")
axes[0].grid()

axes[1].plot(boundary["r"][segment].flatten(), boundary["ti"][segment].flatten())
axes[1].set_title("Ion temperature")
axes[1].set_xlabel("R [m]")
axes[1].set_ylabel("Ti [eV]")
axes[1].grid()

axes[2].plot(boundary["r"][segment].flatten(), boundary["n"][segment].flatten())
axes[2].set_title("Density")
axes[2].set_xlabel("R [m]")
axes[2].set_ylabel("n [m^-3]")
axes[2].grid()

plt.show()


## Recommended takeaway

For new code, prefer this pattern:
- `load_solution(...)`
- `solution.mesh.*` for mesh state
- `solution.fields.*` for assembled arrays
- `solution.views.*` for cached state inspection
- `solution.sample.*` for interpolation workflows
- `solution.pointwise.*` for local diagnostics
- `solution.analysis.*` for integrated physics workflows